# Klines (OHLCV) — Raw Data Explorer

In [ ]:
import pandas as pd

btc_spot = pd.read_parquet('../data/btc_spot_klines.parquet')
btc_perp = pd.read_parquet('../data/btc_perp_klines.parquet')
eth_spot = pd.read_parquet('../data/eth_spot_klines.parquet')
eth_perp = pd.read_parquet('../data/eth_perp_klines.parquet')

## Schema

In [ ]:
btc_spot.dtypes

## First rows — spot

In [ ]:
btc_spot.head(10)

## First rows — perp

In [ ]:
btc_perp.head(10)

## Coverage

In [ ]:
for name, df in [('BTC spot', btc_spot), ('BTC perp', btc_perp), ('ETH spot', eth_spot), ('ETH perp', eth_perp)]:
    print(f"{name}: {len(df)} rows  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

## Gaps check

1h candles — diff should always be 1h.

In [ ]:
gaps = btc_spot['timestamp'].diff().dropna()
unusual = gaps[gaps != pd.Timedelta('1h')]
print(f"BTC spot gaps ≠ 1h: {len(unusual)}")
unusual.head(10)

## Spot vs perp close price comparison (merged)

In [ ]:
merged = btc_spot[['timestamp', 'close']].merge(
    btc_perp[['timestamp', 'close']], on='timestamp', suffixes=('_spot', '_perp')
)
merged['basis_bps'] = (merged['close_perp'] - merged['close_spot']) / merged['close_spot'] * 10000
merged.tail(10)

## Price and volume stats

In [ ]:
btc_spot[['open', 'high', 'low', 'close', 'volume', 'quote_volume']].describe()

## Recent data sample

In [ ]:
btc_spot.tail(24)  # last 24 hours

## Basis vs funding rate

The funding rate is Binance's mechanism to keep the perp anchored to spot. When perp > spot (positive basis), longs pay shorts a positive funding rate. The carry trade collects this payment by going long spot + short perp (delta neutral).

Here we align the 8h funding rate to the basis at the same timestamp to verify the relationship.

In [ ]:
funding = pd.read_parquet('../data/btc_funding_rates.parquet')

# Basis at each hourly candle
basis = btc_spot[['timestamp', 'close']].merge(
    btc_perp[['timestamp', 'close']], on='timestamp', suffixes=('_spot', '_perp')
)
basis['basis_bps'] = (basis['close_perp'] - basis['close_spot']) / basis['close_spot'] * 10000

# Funding settles at 00:00, 08:00, 16:00 UTC — join on those timestamps
# Round funding timestamps to nearest hour to align with kline timestamps
funding['timestamp'] = funding['timestamp'].dt.round('h')
combined = basis.merge(funding, on='timestamp', how='inner')

print(f"Aligned rows (8h funding settlements): {len(combined)}")
print(f"\nCorrelation between basis_bps and funding_rate: {combined['basis_bps'].corr(combined['funding_rate']):.4f}")
combined[['timestamp', 'close_spot', 'close_perp', 'basis_bps', 'funding_rate']].head(20)

## Rolling 1-month regression: basis → funding rate

Runs OLS over a sliding 1-month window (~90 observations at 8h frequency) stepping ~10 days at a time. Shows whether the basis→funding relationship is stable over time or shifts.

In [ ]:
from scipy import stats

WINDOW = 90    # ~1 month at 3 settlements/day
STEP   = 30    # step ~10 days at a time

x = combined['basis_bps'].values
y = combined['funding_rate'].values
ts = combined['timestamp'].values

rows = []
for start in range(0, len(combined) - WINDOW, STEP):
    end = start + WINDOW
    slope, intercept, r, p, _ = stats.linregress(x[start:end], y[start:end])
    rows.append({
        'window_start': ts[start],
        'window_end':   ts[end - 1],
        'slope':        round(slope, 6),
        'intercept':    round(intercept, 6),
        'r':            round(r, 4),
        'r_squared':    round(r**2, 4),
        'p_value':      round(p, 6),
    })

rolling = pd.DataFrame(rows)
rolling

In [ ]:
# Summary: how much does the relationship shift?
print("Rolling r (correlation) stats:")
print(rolling['r'].describe())
print()
print("Rolling slope stats:")
print(rolling['slope'].describe())
print()
print(f"Windows where p_value > 0.05 (not significant): {(rolling['p_value'] > 0.05).sum()}")

## Rolling relationship over time

`r` is the primary metric — unit-free, directly measures how reliably basis predicts funding. The slope needs `× 10000` to be interpretable as "bps of funding per 1 bps of basis" (1.0 = full pass-through).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

dates = pd.to_datetime(rolling['window_end'])
slope_scaled = rolling['slope'] * 10000  # convert to bps-of-funding per bps-of-basis

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# --- top: rolling correlation ---
ax1.plot(dates, rolling['r'], color='steelblue', linewidth=1.5)
ax1.axhline(rolling['r'].mean(), color='steelblue', linestyle='--', alpha=0.5, label=f"mean = {rolling['r'].mean():.3f}")
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_ylabel('Correlation (r)')
ax1.set_title('Rolling 1-month basis → funding rate relationship (BTC)')
ax1.legend()
ax1.set_ylim(-0.1, 1.0)

# --- bottom: slope scaled to bps/bps ---
ax2.plot(dates, slope_scaled, color='darkorange', linewidth=1.5)
ax2.axhline(slope_scaled.mean(), color='darkorange', linestyle='--', alpha=0.5, label=f"mean = {slope_scaled.mean():.3f}")
ax2.axhline(1.0, color='black', linestyle=':', linewidth=0.8, label='full pass-through (1.0)')
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('Slope (bps funding / bps basis)')
ax2.set_xlabel('Window end date')
ax2.legend()

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
fig.autofmt_xdate()

plt.tight_layout()
plt.show()

## What causes the slope spikes?

The slope (pass-through) spikes when the market is **strongly trending** during the 8h settlement window. Normally the basis oscillates and mean-reverts within the window, so the settlement-time snapshot is a noisy proxy for the 8h average. But during a sharp directional move — a rally, a cascade liquidation — the perp premium stays elevated the whole window, so the snapshot IS the average. High slope = high signal quality.

The spikes below check this by overlaying realized volatility on the slope.

In [ ]:
# realized vol: 30d rolling (720 hourly obs), annualized
returns = btc_spot.set_index('timestamp')['close'].pct_change().dropna()
realized_vol = returns.rolling(720).std() * (8760 ** 0.5)

# make window_end UTC-aware so it aligns with the kline index
rolling['window_end'] = pd.to_datetime(rolling['window_end'], utc=True)
rolling['realized_vol'] = [realized_vol.asof(we) for we in rolling['window_end']]

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
dates = rolling['window_end']
slope_scaled = rolling['slope'] * 10000

ax1.plot(dates, slope_scaled, color='darkorange', linewidth=1.5, label='slope (bps/bps)')
ax2.plot(dates, rolling['realized_vol'], color='grey', linewidth=1, alpha=0.6, label='realized vol (30d)')
ax1.set_ylabel('Slope (bps funding / bps basis)', color='darkorange')
ax2.set_ylabel('Realized vol (annualized)', color='grey')
ax1.set_title('Slope spikes vs realized volatility')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f"\nCorrelation between slope and realized vol: {rolling['slope'].corr(rolling['realized_vol']):.4f}")